In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 04: Split 70/15/15 (paper-faithful)

**Paper stage:** Data pre-processing layer — split after augmentation, imputation, wrangling, versioning (paper order: Fig. 1).

We use the augmented+imputed arrays from nb03 (8074 samples). The paper reports 70% training, 15% validation, 15% testing. We perform stratified split by class (CLASS_ORDER: Ilmenite, Rutile, Zircon, Garnet, Monazite, Sillimanite) with fixed SEED=42 for reproducibility. Train/val/test indices and the full split manifest are saved to Drive.

In [2]:
# common setup
import os, sys, random, time
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# [kaggle] Colab Drive mount removed - bsm_env recreates the folder tree

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
for sub in ["data","augmented","arrays","checkpoints","results","figures","versions"]:
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("project root:", PROJECT)


project root: /content/drive/MyDrive/bsm_paper_replication


In [3]:
# split
import os, numpy as np, json, pandas as pd
from sklearn.model_selection import train_test_split

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = os.path.join(PROJECT, "versions")
X = np.load(os.path.join(VER, "X_augmented_imputed.npy"), mmap_mode="r")
y = np.load(os.path.join(VER, "y_augmented_imputed.npy"))
print(X.shape, y.shape)

# stratified split: test 15%, then val 15% of remaining (so train 70%, val 15%, test 15%)
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.15, stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.15/0.85, stratify=y[train_idx], random_state=SEED)
print(len(train_idx), len(val_idx), len(test_idx), len(train_idx)+len(val_idx)+len(test_idx))

# save indices and split arrays
np.save(os.path.join(VER, "train_idx.npy"), train_idx)
np.save(os.path.join(VER, "val_idx.npy"), val_idx)
np.save(os.path.join(VER, "test_idx.npy"), test_idx)

CLASS_ORDER = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]
def stats(arr):
    vc = pd.Series(arr).value_counts().sort_index()
    return {c: int(vc.get(i,0)) for i,c in enumerate(CLASS_ORDER)}

spl = {
  "seed": SEED,
  "split": "70/15/15 stratified by class",
  "shapes": {"train": int(len(train_idx)), "val": int(len(val_idx)), "test": int(len(test_idx))},
  "train_class_counts": stats(y[train_idx]),
  "val_class_counts": stats(y[val_idx]),
  "test_class_counts": stats(y[test_idx])
}
json.dump(spl, open(os.path.join(VER, "split_manifest.json"), 'w'), indent=2)
print(spl)


(8074, 224, 224, 3) (8074,)
5651 1211 1212 8074
{'seed': 42, 'split': '70/15/15 stratified by class', 'shapes': {'train': 5651, 'val': 1211, 'test': 1212}, 'train_class_counts': {'Ilmenite': 955, 'Rutile': 1016, 'Zircon': 908, 'Garnet': 924, 'Monazite': 924, 'Sillimanite': 924}, 'val_class_counts': {'Ilmenite': 204, 'Rutile': 218, 'Zircon': 195, 'Garnet': 198, 'Monazite': 198, 'Sillimanite': 198}, 'test_class_counts': {'Ilmenite': 205, 'Rutile': 218, 'Zircon': 195, 'Garnet': 198, 'Monazite': 198, 'Sillimanite': 198}}


## Save split arrays (X/y split)

In [4]:
# save split arrays
import os, numpy as np
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = os.path.join(PROJECT, "versions")
X = np.load(os.path.join(VER, "X_augmented_imputed.npy"))
y = np.load(os.path.join(VER, "y_augmented_imputed.npy"))
tr = np.load(os.path.join(VER, "train_idx.npy"))
va = np.load(os.path.join(VER, "val_idx.npy"))
te = np.load(os.path.join(VER, "test_idx.npy"))
np.save(os.path.join(VER, "X_train.npy"), X[tr])
np.save(os.path.join(VER, "X_val.npy"), X[va])
np.save(os.path.join(VER, "X_test.npy"), X[te])
np.save(os.path.join(VER, "y_train.npy"), y[tr])
np.save(os.path.join(VER, "y_val.npy"), y[va])
np.save(os.path.join(VER, "y_test.npy"), y[te])
print("saved splits")


saved splits


## Verify and persist

In [5]:
import os, numpy as np, json
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = os.path.join(PROJECT, "versions")
for f in ["X_train.npy","X_val.npy","X_test.npy"]:
    p = os.path.join(VER,f)
    print(f, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else 0)
spl = json.load(open(os.path.join(VER,"split_manifest.json")))
print(spl)


X_train.npy True 3402535040
X_val.npy True 729157760
X_test.npy True 729759872
{'seed': 42, 'split': '70/15/15 stratified by class', 'shapes': {'train': 5651, 'val': 1211, 'test': 1212}, 'train_class_counts': {'Ilmenite': 955, 'Rutile': 1016, 'Zircon': 908, 'Garnet': 924, 'Monazite': 924, 'Sillimanite': 924}, 'val_class_counts': {'Ilmenite': 204, 'Rutile': 218, 'Zircon': 195, 'Garnet': 198, 'Monazite': 198, 'Sillimanite': 198}, 'test_class_counts': {'Ilmenite': 205, 'Rutile': 218, 'Zircon': 195, 'Garnet': 198, 'Monazite': 198, 'Sillimanite': 198}}


In [6]:
import glob, os, shutil, time
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    dst = os.path.join(PROJECT,"notebooks","04_split_70_15_15.ipynb")
    shutil.copy(newest, dst)
    print("copied:", dst)
